# 💬 NLP Course Feedback Analyser
### NTUC LearningHub — Mini Project 2
**Skills:** Text preprocessing · Sentiment Analysis (TextBlob) · Word Frequency · NLP Visualisation  
**Context:** Analyse student course feedback — directly applicable to a training organisation.
---


## 1️⃣ Imports & Feedback Dataset

In [ ]:
%matplotlib inline
import re, collections
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

try:
    from textblob import TextBlob
    TB_OK = True
    print("✅ TextBlob available")
except ImportError:
    TB_OK = False
    print("ℹ  pip install textblob   (running with basic fallback)")

Path("nlp_output").mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")

FEEDBACK = [
    {"id":1,"course":"Python Fundamentals","rating":5,"comment":"Excellent trainer! Very clear explanations and patient with beginners. Loved the hands-on exercises."},
    {"id":2,"course":"Python Fundamentals","rating":4,"comment":"Good course overall. Content was well structured. Could use more advanced examples at the end."},
    {"id":3,"course":"ML Using Python","rating":5,"comment":"Amazing! The machine learning section was incredibly insightful. Real-world datasets made it very practical."},
    {"id":4,"course":"ML Using Python","rating":3,"comment":"Content was okay but the pace was too fast. I struggled to keep up with the ML algorithms section."},
    {"id":5,"course":"Data Analytics Python","rating":5,"comment":"Best Python course I have ever taken. The visualizations chapter was outstanding and very useful for my job."},
    {"id":6,"course":"Data Analytics Python","rating":2,"comment":"Disappointing. The trainer was unprepared and many code examples had errors. Needs improvement."},
    {"id":7,"course":"Python Fundamentals","rating":4,"comment":"Enjoyable learning experience. Would have liked more time on file handling and exception handling topics."},
    {"id":8,"course":"ML Using Python","rating":5,"comment":"Very knowledgeable trainer. Deep learning module was fantastic. Already applying skills at work!"},
    {"id":9,"course":"Data Analytics Python","rating":4,"comment":"Informative and well paced. Seaborn and matplotlib labs were very helpful. Good value course."},
    {"id":10,"course":"Python Fundamentals","rating":1,"comment":"Very basic content. Nothing new for someone with prior programming experience. Too slow."},
    {"id":11,"course":"ML Using Python","rating":4,"comment":"Great practical approach. The sklearn pipelines were new to me and extremely valuable."},
    {"id":12,"course":"Data Analytics Python","rating":5,"comment":"Trainer was engaging and supportive. Interactive dashboards session was particularly impressive."},
]
df = pd.DataFrame(FEEDBACK)
df

## 2️⃣ Sentiment Analysis

In [ ]:
def sentiment(text):
    if not TB_OK:
        return {"polarity":0,"subjectivity":0,"label":"N/A"}
    blob = TextBlob(text)
    pol  = blob.sentiment.polarity
    return {"polarity":round(pol,3),
            "subjectivity":round(blob.sentiment.subjectivity,3),
            "label":"Positive" if pol>0.1 else "Negative" if pol<-0.1 else "Neutral"}

df = df.join(pd.DataFrame(df["comment"].apply(sentiment).tolist()))
df[["id","course","rating","label","polarity","subjectivity"]]

In [ ]:
# Summary stats
print("Avg polarity  :", df["polarity"].mean().round(3))
print("Sentiment counts:")
print(df["label"].value_counts().to_string())

## 3️⃣ Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sentiment pie
counts = df["label"].value_counts()
colors = [{"Positive":"#2ecc71","Neutral":"#f39c12","Negative":"#e74c3c"}.get(l,"grey") for l in counts.index]
axes[0].pie(counts, labels=counts.index, autopct="%1.0f%%", colors=colors,
            startangle=90, wedgeprops={"edgecolor":"white"})
axes[0].set_title("Overall Sentiment", fontweight="bold")

# Avg rating per course
avg_r = df.groupby("course")["rating"].mean().sort_values()
axes[1].barh(avg_r.index, avg_r.values, color=sns.color_palette("Set2",len(avg_r)))
axes[1].set_title("Avg Course Rating", fontweight="bold")
axes[1].set_xlim(0, 5.5)
[axes[1].text(v+0.05, i, f"{v:.1f}", va="center", fontsize=10) for i,v in enumerate(avg_r)]
plt.tight_layout(); plt.show()

## 4️⃣ Keyword Frequency Analysis

In [ ]:
STOP = {"the","a","an","and","is","it","to","was","i","for","of","in","very",
        "with","be","this","that","at","by","my","me","on","are","had","but",
        "more","have","we","as","so","not","too"}

def preprocess(text):
    return [t for t in re.findall(r"[a-z]+", text.lower())
            if t not in STOP and len(t)>2]

all_tokens = [t for row in df["comment"] for t in preprocess(row)]
top15 = collections.Counter(all_tokens).most_common(15)
words, freqs = zip(*top15)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(words, freqs, color=sns.color_palette("husl", len(words)), edgecolor="white")
ax.set_title("Top 15 Keywords in Course Feedback", fontweight="bold", fontsize=13)
ax.set_xlabel("Keyword"); ax.set_ylabel("Frequency")
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.show()

## 5️⃣ Polarity Distribution by Course

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="course", y="polarity", palette="Set2", ax=ax)
ax.axhline(0, color="red", linestyle="--", alpha=0.6)
ax.set_title("Sentiment Polarity Distribution by Course", fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("Polarity Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
plt.tight_layout(); plt.show()